In [11]:
import chromadb
from sentence_transformers import SentenceTransformer #  pre-trained model that converts text into embeddings
from openai import OpenAI
import uuid # uuid is just a helper for unique IDs in Chroma.
from google.colab import files


In [ ]:
# This block sets up a secure connection (client) to Groq’s LLM using your API key so your code can ask questions and get answers.
GROQ_API_KEY = "API_KEY_HERE"

#The API key is passed to the OpenAI-compatible client so it can authenticate requests.
#client is a Python object that represents a connection to the Groq API
# client is used to:
# Send the retrieved context + question to the LLM
# Receive the generated answer
client = OpenAI(
    api_key=GROQ_API_KEY,
    base_url="https://api.groq.com/openai/v1" #The API endpoint URL. For Groq,
)


In [13]:
# -----------------------------
# Initialize Embedding Model
# -----------------------------
# Loads a pre-trained model that converts text into numerical vectors (embeddings).
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

def get_embedding(text):
    return embedding_model.encode(text).tolist()


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [14]:
# -----------------------------
# Initialize Chroma DB (In-Memory)
# -----------------------------
chroma_client = chromadb.Client()  # creates a connection to the Chroma vector database.
# A collection is like a table or bucket in Chroma DB where you store related embeddings.
collection = chroma_client.get_or_create_collection(name="documents")


In [19]:
# -----------------------------
# Chunking Function
# -----------------------------
# Chunk_size = Number of characters in each chunk (500 here).
# overlap = How much each chunk overlaps with the previous chunk (100 here).
def chunk_text(text, chunk_size=500, overlap=100):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks

# -----------------------------
# Upload Text Files
# -----------------------------
print("📂 Upload your .txt files")
uploaded_files = files.upload()

# -----------------------------
# Index Documents
# -----------------------------
#Loop through all uploaded files.
for filename in uploaded_files.keys():
    text = uploaded_files[filename].decode("utf-8") # Convert bytes → string so Python can process it.
    chunks = chunk_text(text)

    for chunk in chunks:
        collection.add(
            ids=[str(uuid.uuid4())],
            embeddings=[get_embedding(chunk)],
            documents=[chunk]
        )

print("✅ Documents indexed successfully!")

📂 Upload your .txt files


Saving rag_pdf.txt to rag_pdf (1).txt
✅ Documents indexed successfully!


In [22]:
# -----------------------------
# Question-Answer Loop
# -----------------------------
def ask_question(question):
  #Converts the user’s question into an embedding vector using the embedding model.
    query_embedding = get_embedding(question)

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=3
    )

    retrieved_chunks = results["documents"][0]

    if not retrieved_chunks:
        return "No relevant information found."

    context = "\n\n".join(retrieved_chunks)

    prompt = f"""
Answer the question using ONLY the context below.
If the answer is not found, say: "Not found in document."

Context:
{context}

Question:
{question}
"""

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}],
        temperature=0  # 0 → deterministic answers (less randomness).
    )

    return response.choices[0].message.content

# -----------------------------
# Interactive Loop
# -----------------------------
while True:
    question = input("\n🤓 Ask a question (type 'exit' to stop): ")

    if question.lower() == "exit":
        break

    answer = ask_question(question)
    print("\n🧠 Answer:")
    print(answer)


🤓 Ask a question (type 'exit' to stop): What are the four main OOP principles in Java?

🧠 Answer:
The four main OOP principles in Java are:

1. Encapsulation
2. Inheritance
3. Polymorphism
4. Abstraction

🤓 Ask a question (type 'exit' to stop): In which year was Java first released?

🧠 Answer:
Not found in document.

🤓 Ask a question (type 'exit' to stop): What are the two categories of data types in Java?

🧠 Answer:
Two categories of data types in Java are: 

1. Primitive data types 
2. Non-primitive data types

🤓 Ask a question (type 'exit' to stop): Which keyword is used for inheritance in Java?

🧠 Answer:
extends

🤓 Ask a question (type 'exit' to stop): What does IDE stand for?

🧠 Answer:
Not found in document.

🤓 Ask a question (type 'exit' to stop): What is the use of comments in programming?

🧠 Answer:
Not found in document.

🤓 Ask a question (type 'exit' to stop): exit
